In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction


In [3]:
import json
import pandas as pd
import os
import numpy as np

# =============================================================================
# STEP 0: Load Data
# =============================================================================
# Use the base_folder to construct the absolute path
data_path = os.path.join(base_folder, "data", "train.csv")
df = pd.read_csv(data_path)

# Drop columns not needed for prediction or already handled
df = df.drop(columns=["satisfaction", "Unnamed: 0", "id"], errors="ignore")

# =============================================================================
# STEP 1: Define Features Correctly
# =============================================================================
# 1. True Numerical (continuous/delay)
TRUE_NUMERICAL = [
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

# 2. Ordinal Ratings (1-5) - Treat as Numerical for min/max stats
RATING_FEATURES = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# All features we want min/max statistics for
NUMERICAL_ALL = TRUE_NUMERICAL + RATING_FEATURES

# 3. True Categorical (non-numeric text)
CATEGORICAL = [
    'Gender', 'Customer Type', 'Type of Travel', 'Class'
]

schema = {"numerical": {}, "categorical": {}}

# =============================================================================
# STEP 2: Analyze Numerical Features (including ratings)
# =============================================================================
for col in NUMERICAL_ALL:
    data_for_stats = df[col].dropna()

    schema["numerical"][col] = {
        "min": float(data_for_stats.min()),
        "max": float(data_for_stats.max()),
        "mean": float(data_for_stats.mean()),
        "median": float(data_for_stats.median()),
        "type": "rating (1-5)" if col in RATING_FEATURES else "delay/distance"
    }

# =============================================================================
# STEP 3: Analyze Categorical Features
# =============================================================================
for col in CATEGORICAL:
    schema["categorical"][col] = {
        "unique_values": sorted(df[col].dropna().unique().tolist()),
        "value_counts": df[col].value_counts().to_dict(),
    }

# =============================================================================
# STEP 4: Save Schema Safely
# =============================================================================
output_file = os.path.join(base_folder, "data", "data_schema.json")

# Ensure the 'data' directory exists
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, "w") as f:
    json.dump(schema, f, indent=2)

print(f"✓ data_schema.json generated and saved to: {output_file}")

✓ data_schema.json generated and saved to: /content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/data/data_schema.json
